In [149]:
from sklearn.linear_model import LogisticRegression
import numpy as np
import pickle

In [150]:
root_path = "/home/stefan/ioai-prep/kits/dataset_gender_biases_shall_not_pass"
seed = 42

# Data

In [151]:
with open(f"{root_path}/train_data.pkl", "rb") as f:
    data = pickle.load(f)

{k: data[k].shape for k in data.keys()}

{'train_image_embeddings': (15000, 512),
 'gender_labels': (15000,),
 'test_image_embeddings': (15000, 512),
 'compression': (4, 512)}

In [152]:
train_image_embeddings = data["train_image_embeddings"]
gender_labels = data["gender_labels"]
test_image_embeddings = data["test_image_embeddings"]

# Theory Background

### Vector Spaces, Span, and Subspaces

The CLIP embeddings live in a real vector space $\mathbb{R}^d$, where $d$ is the embedding dimension (e.g., 512 for CLIP ViT-B/32, 768 for ViT-L/14).

Given a non-zero vector $g \in \mathbb{R}^d$, its **span** defines a 1-dimensional subspace (a line through the origin):

$$\text{span}\{g\} = \{ \alpha g \mid \alpha \in \mathbb{R} \}$$

The set of all vectors **orthogonal** to $g$ forms a $(d-1)$-dimensional subspace:

$$g^{\perp} = \{ x \in \mathbb{R}^d \mid x \cdot g = 0 \}$$

These two subspaces are complementary: $\mathbb{R}^d = \text{span}\{g\} \oplus g^{\perp}$.
Every vector $x \in \mathbb{R}^d$ can be uniquely decomposed as $x = x_{\parallel} + x_{\perp}$ where $x_{\parallel} \in \text{span}\{g\}$ and $x_{\perp} \in g^{\perp}$.

### Orthogonal Projection

Given a vector $x \in \mathbb{R}^d$ and a direction defined by a non-zero vector $g \in \mathbb{R}^d$, the orthogonal projection of $x$ onto $g$ decomposes $x$ into two perpendicular components:

$$x = x_{\parallel} + x_{\perp}$$

where $x_{\parallel} \parallel g$ (parallel to $g$) and $x_{\perp} \perp g$ (orthogonal to $g$).

---

#### Derivation of the vector formula

Since $x_{\parallel}$ is parallel to $g$, it must be a scalar multiple: $x_{\parallel} = \alpha g$ for some $\alpha \in \mathbb{R}$.

The residual $x_{\perp} = x - \alpha g$ must be orthogonal to $g$ by definition:

$$(x - \alpha g) \cdot g = 0$$

Expanding:

$$x \cdot g - \alpha (g \cdot g) = 0$$

Since $g \cdot g = \|g\|^2$:

$$\alpha = \frac{x \cdot g}{\|g\|^2}$$

Therefore the **orthogonal projection** of $x$ onto $g$ is:

$$\text{proj}_g(x) = \frac{x \cdot g}{\|g\|^2}\,g$$

When $g$ is a **unit vector** ($\|g\| = 1$), the formula simplifies to:

$$\text{proj}_g(x) = (x \cdot g)\,g$$

The **residual** (component orthogonal to $g$) is:

$$x_{\perp} = x - \text{proj}_g(x) = x - \frac{x \cdot g}{\|g\|^2}\,g$$

#### Geometric interpretation

$P_g x$ gives the closest point to $x$ on the line $\text{span}\{g\}$. For any scalar $\alpha$, minimizing the squared distance:

$$\|x - \alpha g\|^2 = \|x\|^2 - 2\alpha(x \cdot g) + \alpha^2\|g\|^2$$

Differentiating w.r.t. $\alpha$ and setting to zero: $-2(x \cdot g) + 2\alpha\|g\|^2 = 0\ \Rightarrow\ \alpha = \frac{x \cdot g}{\|g\|^2}$, confirming the formula.

In the debiasing context: removing $P_g x$ from $x$ erases all information along direction $g$, so a linear classifier with weights parallel to $g$ sees the same value (zero) for every debiased embedding.

# Solution

The gender bias axis $g$ (a unit vector) is found by one of two methods:

In [ ]:
male_mean = train_image_embeddings[gender_labels == 1].mean(axis=0)
female_mean = train_image_embeddings[gender_labels == 0].mean(axis=0)

w = male_mean - female_mean
bias_axis = w / np.linalg.norm(w)

**1. Mean Difference**

$$g = \frac{\mu_{\text{male}} - \mu_{\text{female}}}{\|\mu_{\text{male}} - \mu_{\text{female}}\|}$$

where $\mu_{\text{male}}, \mu_{\text{female}} \in \mathbb{R}^d$ are the mean CLIP embeddings for each gender class.

In [ ]:
clf = LogisticRegression(solver="liblinear", C=0.1, random_state=42)
clf.fit(train_image_embeddings, gender_labels)

w = clf.coef_.flatten()
lr_bias_axis = w / np.linalg.norm(w)

**2. Logistic Regression**

A linear classifier $f(x) = \sigma(w^\top x + b)$ is trained to predict gender from the embedding. The weight vector $w$ is normal to the decision boundary, so it encodes the direction that best separates the two classes:

$$g = \frac{w}{\|w\|}$$

Both methods identify a 1-dimensional subspace (a line through the origin) in $\mathbb{R}^d$ that captures gender-related variation in the CLIP embeddings.

In [ ]:
def debias_embeddings(embd: np.array):
    g = bias_axis / np.linalg.norm(bias_axis)

    projections = (embd @ g).reshape(-1, 1)

    debiased = embd - projections * g
    return debiased

### Debiasing via Orthogonal Projection

The debiasing operation removes the component of each embedding along the gender bias axis:

$$\hat{x} = x - (x \cdot g)\, g$$

This is the **orthogonal projection** of $x$ onto the $(d-1)$-dimensional subspace orthogonal to $g$. The result $\hat{x}$ satisfies $\hat{x} \cdot g = 0$, meaning the embedding has zero component along the gender direction.

**Key property:** A linear classifier with weight vector $w$ parallel to $g$ cannot distinguish gender from $\hat{x}$:

$$w^\top \hat{x} = w^\top (x - (x \cdot g)g) = w^\top x - (x \cdot g)(w^\top g)$$

Since $g \parallel w$ and $\|g\| = 1$, we have $w = \|w\| \cdot g$, and thus $w^\top g = \|w\|$. The dot product with $\hat{x}$:

$$w^\top \hat{x} = w^\top x - \|w\|(x \cdot g) = \|w\|(x \cdot g) - \|w\|(x \cdot g) = 0$$

So the linear gender signal is completely removed.

In [155]:
debiased_train_image_embeddings = debias_embeddings(train_image_embeddings)
debiased_test_image_embeddings = debias_embeddings(test_image_embeddings)

# Submission

In [156]:
def generate_submission(
    debiased_train_image_embeddings,
    debiased_test_image_embeddings,
    compression=data["compression"],
):
    train_compression = debiased_train_image_embeddings @ compression.T
    test_compression = debiased_test_image_embeddings @ compression.T
    with open(f"{root_path}/submission.pkl", "wb") as f:
        pickle.dump({
            "train_compression": train_compression,
            "test_compression": test_compression,
        }, f)

generate_submission(debiased_train_image_embeddings, debiased_test_image_embeddings)